In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# Setup
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) 
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

from data_loader import download_and_save, load_local_dataset
from audit_text import *
from audit_audio import *
from utils import load_config, ensure_dir

In [5]:
# Load configs
paths = load_config("../configs/paths.yaml")
audit_cfg = load_config("../configs/audit_config.yaml")

FIGURES_PATH = os.path.join(PROJECT_ROOT, paths["outputs"]["figures"])
REPORTS_PATH = os.path.join(PROJECT_ROOT, paths["outputs"]["reports"])
ensure_dir(FIGURES_PATH)
ensure_dir(REPORTS_PATH)

WindowsPath('C:/Users/MSI/Desktop/ASR/asr_tn_pielines/outputs/reports')

In [6]:
# Load dataset
DATA_PATH = os.path.join(PROJECT_ROOT, paths["data"]["raw"])
# dataset = download_and_save("linagora/linto-dataset-audio-ar-tn", DATA_PATH)
dataset = load_local_dataset(DATA_PATH)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['audio_id', 'audio', 'segments', 'transcript'],
    num_rows: 20895
})
Dataset({
    features: ['audio_id', 'audio', 'segments', 'transcript'],
    num_rows: 799
})


In [7]:
from datasets import Audio

train_dataset = train_dataset.cast_column("audio", Audio(decode=False))
test_dataset = test_dataset.cast_column("audio", Audio(decode=False))

In [8]:
def segment_audio(speech, sr, segments):
    segmented_data = []

    for seg in segments:
        start_sample = round(seg["start"] * sr)
        end_sample = round(seg["end"] * sr)
        start_sample = max(0, start_sample)
        end_sample = min(len(speech), end_sample)
        chunk = speech[start_sample:end_sample]

        segmented_data.append({
            "audio": chunk,
            "sampling_rate": sr,
            "text": seg["transcript"],
            "start": seg["start"],
            "end": seg["end"]
        })

    return segmented_data

In [9]:
list(test_dataset['audio_id'])[:5]

['amenyKH_0000001_chunk_0000288_001_chunk_288',
 'amenyKH_0000002_chunk_0000076_002_chunk_76',
 'amenyKH_0000002_chunk_0000085_002_chunk_85',
 'amenyKH_0000002_chunk_0000107_002_chunk_107',
 'amenyKH_0000004_chunk_0000010_004_chunk_10']

In [29]:
from IPython.display import Audio, display

def find_and_listen(audio, index, offset=5, train=True):
    if train:
        dataset = train_dataset
    else:
        dataset = test_dataset
    x=0
    for i, j in enumerate(dataset):
        if j['audio_id'] == audio:
            x=i
            break
    sample = dataset[x]
    
    speech, sr, shape = load_audio_raw(sample)
    
    segments = sample["segments"]
    segments = sorted(segments, key=lambda x: x["start"])

    for i, seg in enumerate(segments[index:index+offset]):
        start = round(seg["start"] * sr)
        end = round(seg["end"] * sr)
    
        chunk = speech[start:end]
        if chunk.ndim > 1:
            chunk = chunk.mean(axis=1)
        print(f"\n--- Segment {i} ---")
        print(f"[{seg['start']:.2f}s → {seg['end']:.2f}s]")
        if "c'إي" in seg["transcript"]:
            print("true")
        print("Transcript:", seg["transcript"])
        print(f"Duration: {(end - start)/sr:.2f}s")
    
        display(Audio(data=chunk, rate=sr))
    
# find_and_listen('amenyKH_0000002_chunk_0000076_002_chunk_76', 0, 10)

lisst = ['amenyKH_0000021_0000008_21_8_0',
 'amenyKH_0000026_0000032_0000001_26_32_1_0',
 'amenyKH_0000026_0000036_0000001_26_36_1_0',
 'amenyKH_Hkayet_Tounsia_S0000001_Episode_0000004_0000019_0000012_0002016_Partie_0000002_Sync0000096_no_speaker_0000700.0000027_0000700.0000055_Hkayet_Tounsia_S01_Episode_04_19_12_2016_Partie_02_Sync096_no_speaker_700.27_700.55_0',
 'amenyKH_Hkayet_Tounsia_S0000001_Episode_0000017_0000020_0000003_0002017_Partie_0000003_Sync0000014_no_speaker_0000863.0000045_0000863.0000090_Hkayet_Tounsia_S01_Episode_17_20_03_2017_Partie_03_Sync014_no_speaker_863.45_863.90_0',]
for i in lisst:
    aud_id, chunk = i.rsplit('_', 1)
    find_and_listen(aud_id, int(chunk), 1, True)



--- Segment 0 ---
[0.00s → 1.47s]
Transcript: أهلا بيكم
Duration: 1.47s



--- Segment 0 ---
[0.00s → 1.36s]
Transcript: أفف فوالا
Duration: 1.36s



--- Segment 0 ---
[0.00s → 1.31s]
Transcript: أفف باهي
Duration: 1.31s



--- Segment 0 ---
[0.00s → 1.28s]
Transcript: امراة
Duration: 1.28s



--- Segment 0 ---
[0.00s → 1.46s]
Transcript: الرابع
Duration: 1.46s


In [12]:
import requests
import io
import soundfile as sf
from IPython.display import Audio, display

def hf_predict_http_bytes(audio_bytes):
    """
    Send WAV audio bytes as file to HF Space HTTP API and return prediction.
    """
    url = "https://salahza-tunisian-speech-recognition.hf.space/run/predict"

    # Prepare in-memory file
    files = {
        "file": ("audio.wav", audio_bytes, "audio/wav")
    }

    response = requests.post(url, files=files)

    # Check response status
    if response.status_code != 200:
        raise RuntimeError(f"Request failed: {response.status_code}\n{response.text}")

    # Parse JSON
    result = response.json()
    return result["data"][0]

In [13]:
from IPython.display import Audio, display
import numpy as np
import soundfile as sf
import io

def listen_and_predict(audio_id, index=0, offset=5):
    """
    Listen to audio segments, show original transcript, 
    get transcription from HF demo (HTTP API, in-memory).
    """
    # Find the sample
    x = 0
    for i, j in enumerate(train_dataset):
        if j['audio_id'] == audio_id:
            x = i
            break
    sample = train_dataset[x]
    
    # Load audio
    speech, sr, shape = load_audio_raw(sample)
    segments = sorted(sample["segments"], key=lambda x: x["start"])
    
    for i, seg in enumerate(segments[index:index+offset]):
        start = round(seg["start"] * sr)
        end = round(seg["end"] * sr)
        chunk = speech[start:end]
        
        print(f"\n--- Segment {i} ---")
        print(f"[{seg['start']:.2f}s → {seg['end']:.2f}s]")
        print("Original Transcript:", seg["transcript"])
        print(f"Duration: {(end - start)/sr:.2f}s")
        display(Audio(data=chunk, rate=sr))
        
        # Convert to in-memory WAV bytes
        buf = io.BytesIO()
        sf.write(buf, chunk, sr, format="WAV")
        buf.seek(0)
        
        # Get prediction
        import time
        time.sleep(1)
        try:
            prediction = hf_predict_http_bytes(buf.read())
        except RuntimeError as e:
            print("HF API error:", e)
            prediction = "[API rate limit or error]"

In [14]:
listen_and_predict('youtube_UIrAnY3HH0A', 0, 1)


--- Segment 0 ---
[5.58s → 9.94s]
Original Transcript: الضروف كانت صعيبه أمي كانت تخدم علينا باباي كيف كيف زادا﻿ كان يخدم علينا
Duration: 4.35s


HF API error: Request failed: 500
<!DOCTYPE html>
<html class="">
<head>
    <meta charset="utf-8"/>
    <meta
            name="viewport"
            content="width=device-width, initial-scale=1.0, user-scalable=no"
    />
    <meta
            name="description"
            content="We’re on a journey to advance and democratize artificial intelligence through open source and open science."
    />
    <meta property="fb:app_id" content="1321688464574422"/>
    <meta name="twitter:card" content="summary_large_image"/>
    <meta name="twitter:site" content="@huggingface"/>
    <meta
            property="og:title"
            content="Hugging Face – The AI community building the future."
    />
    <meta property="og:type" content="website"/>

    <title>Hugging Face – The AI community building the future.</title>
    <style>
        body {
            margin: 0;
        }

        main {
            background-color: white;
            min-height: 100vh;
            padding: 7rem 1rem 

# search for similar words

In [ ]:
def find_similar_words_rapidfuzz(
    column,
    w_len=10,
    dist_input=1,
    limit=10000000,
    min_freq=2
):
    """
    High-performance fuzzy matching using RapidFuzz + delete index.

    Parameters
    ----------
    column : Iterable[str]
    w_len : int
    dist_input : int
        Exact Levenshtein distance to match
    limit : int
    min_freq : int

    Returns
    -------
    list of tuples: (word1, word2, distance)
    """

    from collections import Counter, defaultdict
    from rapidfuzz.distance import Levenshtein

    # ── 1. Build vocab ───────────────────────────────────────────────
    words = [
        w
        for text in column
        if isinstance(text, str)
        for w in text.split()
        if len(w) == w_len
    ]

    counts = Counter(words)
    vocab = [w for w, c in counts.items() if c >= min_freq]

    # ── 2. Generate deletion keys (cached for speed) ─────────────────
    delete_cache = {}

    def generate_deletes(word):
        if word in delete_cache:
            return delete_cache[word]

        deletes = set()

        def _rec(w, d):
            if d == 0:
                return
            for i in range(len(w)):
                w_del = w[:i] + w[i+1:]
                if w_del not in deletes:
                    deletes.add(w_del)
                    _rec(w_del, d - 1)

        _rec(word, dist_input)
        delete_cache[word] = deletes
        return deletes

    # ── 3. Build delete index ────────────────────────────────────────
    delete_index = defaultdict(set)

    for w in vocab:
        for d in generate_deletes(w):
            delete_index[d].add(w)

    # ── 4. Candidate matching ───────────────────────────────────────
    seen = set()
    pairs = []

    for w in vocab:
        candidates = set()
        for d in generate_deletes(w):
            candidates |= delete_index[d]

        for c in candidates:
            if w >= c:
                continue

            pair_id = (w, c)
            if pair_id in seen:
                continue
            seen.add(pair_id)

            # RapidFuzz distance (VERY fast)
            computed_dist = Levenshtein.distance(w, c)

            if computed_dist == dist_input:
                pairs.append((w, c, computed_dist))

    # ── 5. Sort (distance + frequency) ──────────────────────────────
    pairs.sort(key=lambda x: (x[2], -(counts[x[0]] + counts[x[1]])))

    if limit:
        pairs = pairs[:limit]

    return pairs

pairs = find_similar_words_rapidfuzz(
    df_train['transcript_cs'],
    w_len=4,
    dist_input=1,
    min_freq=5
)
print(len(pairs))
for w1, w2, d in pairs[:100]:
    print(f"{w1}	{w2}")

In [ ]:
from tqdm.auto import tqdm
import numpy as np

def compute_audio_channels(hf_dataset) -> dict[str, int]:
    """
    Loop over raw recordings and count mono vs multi-channel audio.
    """

    counts = {
        "mono": 0,
        "multi_channel": 0,
        "invalid": 0,
    }

    for i in tqdm(range(len(hf_dataset)), desc="Scanning audio channels"):
        audio = hf_dataset[i]["audio"]
        audio_bytes = audio.get("bytes")

        if not audio_bytes:
            counts["invalid"] += 1
            continue

        try:
            waveform, sr = decode_audio_bytes(audio_bytes)
        except Exception:
            counts["invalid"] += 1
            continue

        if not isinstance(waveform, np.ndarray):
            counts["invalid"] += 1
            continue

        if waveform.ndim == 1:
            counts["mono"] += 1
        elif waveform.ndim == 2:
            counts["multi_channel"] += 1
            print(i)
        else:
            counts["invalid"] += 1

    return counts

stats = compute_audio_channels(test_dataset)
print(stats)